# 01 · Setup, download and look at the data
Sets up Colab, downloads the two handwriting datasets to Google Drive, and **prints what is really in them**
(file types, folders, label files, columns). We write the loading code only after seeing this, because
we haven't yet checked the real files.

**No GPU needed for this notebook.** Leave the runtime on CPU to save your free GPU time.

> Reminder: this project is an assistive tool, not a medical decision-maker.
> Every output must be checked by a human against the original prescription.

### Setup
Mounts Google Drive (where data, checkpoints and results are kept), pulls the latest code from GitHub
and installs the extra packages. Run it every time you open the notebook.
If the repo is private, add a Colab Secret named `GITHUB_TOKEN` (key icon in the left sidebar).

In [ ]:
import os, sys, importlib
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/AlstonPR/PrescriptionScanner-ml.git"
REPO_DIR = "/content/PrescriptionScanner-ml"

try:  # use the token for a private repo; public repos work without it
    from google.colab import userdata
    REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
if not os.path.exists(f"{REPO_DIR}/src"):
    raise RuntimeError("Could not get the repo. If it is private, add a GITHUB_TOKEN secret and re-run.")
%cd {REPO_DIR}
!pip install -q -r requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
sys.path_importer_cache.clear()
importlib.invalidate_caches()

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none (fine for this notebook; Runtime > Change runtime type > T4 GPU for training)")

## Step 1 · Download the datasets to Drive
Downloads only happen once. Later runs find the files already on Drive.

In [ ]:
import shutil
from src import config, data

config.BD_DIR.mkdir(parents=True, exist_ok=True)
config.RX_DIR.mkdir(parents=True, exist_ok=True)

# --- Dataset 1: Kaggle (automatic, with a manual fallback) ---
data.unzip_all(config.BD_DIR)
if not any(p.is_file() for p in config.BD_DIR.rglob("*")):
    try:
        import kagglehub
        path = kagglehub.dataset_download("mamun1113/doctors-handwritten-prescription-bd-dataset")
        shutil.copytree(path, config.BD_DIR, dirs_exist_ok=True)
        print("BD dataset copied to", config.BD_DIR)
    except Exception as e:
        print("Automatic Kaggle download failed:", e)
        print("Fallback: download the zip from kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset,")
        print(f"upload it to Google Drive at {config.BD_DIR}, then re-run this cell (it unzips automatically).")
else:
    print("BD dataset already on Drive.")

# --- Dataset 2: Mendeley (manual: Mendeley has no simple download API) ---
data.unzip_all(config.RX_DIR)
if not any(p.is_file() and p.suffix != ".zip" for p in config.RX_DIR.rglob("*")):
    print("\nRxHandBD is not on Drive yet:")
    print("  1. Open https://data.mendeley.com/datasets/dsb5r6vskg and click 'Download All'.")
    print(f"  2. Upload the zip to Google Drive at {config.RX_DIR}")
    print("  3. Re-run this cell; it unzips automatically.")
else:
    print("RxHandBD already on Drive.")

## Step 2 · What is inside each dataset?
Prints file types, folders with image counts, example file names, and the columns of any label tables.
**Copy this whole output back to me.** It tells me how to write the loader (where the labels and the
writer / prescription IDs are).

In [ ]:
data.unzip_all(config.BD_DIR); data.unzip_all(config.RX_DIR)   # in case you just uploaded a zip
data.describe_dataset(config.BD_DIR)
print()
data.describe_dataset(config.RX_DIR)

## Step 3 · Look at a few images
A quick look at what the model will see. Check that the images are word crops and not whole pages.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(config.SEED)
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for row, (name, root) in zip(axes, [("bd", config.BD_DIR), ("rxhandbd", config.RX_DIR)]):
    imgs = [p for p in root.rglob("*") if p.suffix.lower() in data.IMAGE_EXTS]
    for ax, p in zip(row, random.sample(imgs, min(4, len(imgs)))):
        ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
        ax.set_title(f"{name}: {p.name}", fontsize=8)
plt.tight_layout(); plt.show()

## ✅ Check before moving on
- Step 1: both datasets are on Drive. RxHandBD is a manual upload, and the cell tells you how.
- Step 2: the image counts are close to what we expect (about 4,680 for BD and 5,578 for RxHandBD).
  Note where the **labels** are (a csv/xlsx file, file names, or folder names) and whether there is a
  **writer or prescription ID** anywhere. We need that ID to split without leakage.
- Step 3: the images are single word crops.

Send me the printed output of Step 2 (and tell me what you see in Step 3).
Next: **02_clean_and_split.ipynb**, written from the real file layout.